# Visualizzazione A* su Mappa Reale - Mumbai
## Percorso ottimale per consegne Amazon a Mumbai con cache condivisa

Questo notebook applica **A*** su **Mumbai** usando la stessa logica di `search.ipynb`.  
✅ **Cache OSMnx condivisa**: se hai già eseguito search.ipynb, il download sarà **istantaneo**!

**Workflow:**
1. Filtra dataset per Mumbai (18.90-19.30°N, 72.70-73.00°E) 
2. Seleziona consegna ≥2 km per visualizzazione ottimale
3. Scarica rete stradale con `osmnx` (cache attiva)
4. Applica A* con euristica Haversine su grafo reale
5. Visualizza con mappa interattiva Folium

## 1. Installazione e Import delle Librerie Geospaziali

In [1]:
# Installazione delle librerie (eseguire solo se non già installate)
import subprocess, sys

for pkg in ["osmnx", "folium", "networkx"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# ═══════════════════════════════════════════════════════════════
# Import librerie
# ═══════════════════════════════════════════════════════════════
import osmnx as ox
import networkx as nx
import folium
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import radians, sin, cos, sqrt, atan2

print(f"osmnx    : {ox.__version__}")
print(f"networkx : {nx.__version__}")
print(f"folium   : {folium.__version__}")
print("✅ Tutte le librerie importate correttamente.")

osmnx    : 2.1.0
networkx : 3.6.1
folium   : 0.20.0
✅ Tutte le librerie importate correttamente.


## 2. Filtro Mumbai e Selezione Ordine Ottimale

In [2]:
# ═══════════════════════════════════════════════════════════════
# Caricamento del dataset - FOCUS MUMBAI
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("../Preprocessing/amazon_delivery_final.csv")

# Funzione haversine per calcolare distanze
def haversine_km(lat1, lon1, lat2, lon2):
    """Distanza haversine in chilometri."""
    from math import radians, sin, cos, sqrt, atan2
    R = 6371.0  # raggio terrestre medio in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── FILTRO MUMBAI (stesso di search.ipynb) ──
mumbai = df[
    (df['Store_Latitude'].between(18.90, 19.30)) &
    (df['Store_Longitude'].between(72.70, 73.00))
].copy()

if len(mumbai) > 0:
    city_label = "Mumbai"
    subset = mumbai
    print(f"✓ Trovate {len(mumbai)} righe nell'area di Mumbai.")
else:
    # Fallback: Indore (molto presente nel dataset)
    subset = df[
        (df['Store_Latitude'].between(22.60, 22.85)) &
        (df['Store_Longitude'].between(75.70, 75.95))
    ].copy()
    city_label = "Indore"
    print(f"✗ Nessun punto a Mumbai. Fallback → Indore ({len(subset)} righe).")

# Calcola distanza Store–Drop per ogni riga
subset['dist_km'] = subset.apply(
    lambda r: haversine_km(r['Store_Latitude'], r['Store_Longitude'],
                           r['Drop_Latitude'], r['Drop_Longitude']), axis=1)

# Seleziona un ordine con distanza ≥ 2 km (per visualizzare un percorso significativo)
long_deliveries = subset[subset['dist_km'] >= 2.0].sort_values('dist_km')

if len(long_deliveries) == 0:
    # Se non ci sono consegne ≥ 2 km, prendi la più lunga disponibile
    sample = subset.sort_values('dist_km', ascending=False).iloc[0]
    print(f"⚠ Nessuna consegna ≥ 2 km. Uso la più lunga: {sample['dist_km']:.2f} km")
else:
    # Prendi una consegna intermedia (~3-5 km) per buona visualizzazione
    sample = long_deliveries.iloc[len(long_deliveries) // 2]

# Punto A (Store) e Punto B (Drop-off)
store_lat, store_lon = sample['Store_Latitude'], sample['Store_Longitude']
drop_lat,  drop_lon  = sample['Drop_Latitude'],  sample['Drop_Longitude']

print(f"\n╔══════════════════════════════════════════════════════════════════╗")
print(f"║  Città              : {city_label}")
print(f"║  Ordine selezionato : {sample['Order_ID']}")
print(f"║  Punto A (Store)    : ({store_lat:.6f}, {store_lon:.6f})")
print(f"║  Punto B (Drop-off) : ({drop_lat:.6f}, {drop_lon:.6f})")
print(f"║  Distanza aerea     : {sample['dist_km']:.2f} km")
print(f"║  Traffic Jam        : {int(sample['Traffic_Jam'])}")
print(f"║  Traffic High       : {int(sample['Traffic_High'])}")
print(f"╚══════════════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║  Ordine selezionato : uhpp623003037
║  Punto A (Store)    : (18.530963, 73.828972)
║  Punto B (Drop-off) : (18.590963, 73.888972)
║  Traffic Jam        : 1
║  Traffic High       : 0
╚══════════════════════════════════════════════════════╝


## 3. Download Ottimizzato con Cache Condivisa

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Download rete stradale da OpenStreetMap - STESSO METODO DI search.ipynb
# ═══════════════════════════════════════════════════════════════
# Centro e raggio: copriamo l'area Store–Drop con un margine  
center_lat = (store_lat + drop_lat) / 2
center_lon = (store_lon + drop_lon) / 2
radius_m   = max(sample['dist_km'] * 1000 * 0.8, 2500)  # almeno 2.5 km

print(f"Download grafo {city_label}:")
print(f"  Centro: ({center_lat:.4f}, {center_lon:.4f})")
print(f"  Raggio: {radius_m:.0f} m")

# ⚠️ CACHE: Se hai già eseguito search.ipynb per Mumbai, questo sarà VELOCE! 
print(f"\n⏳ Download rete stradale (cache attiva)...")

# Configurazione cache OSMnx (stesso di search.ipynb)
import osmnx as ox
ox.settings.use_cache = True
ox.settings.log_console = False

# STESSO METODO DI search.ipynb: graph_from_point invece di bbox
G = ox.graph_from_point((center_lat, center_lon), dist=radius_m, network_type='drive')
print("✅ Rete stradale scaricata con successo!")

# Statistiche di base
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print(f"\n📊 Statistiche del grafo {city_label}:")
print(f"   Nodi  : {num_nodes:,}")
print(f"   Archi : {num_edges:,}") 
print(f"📁 Cache riutilizzata se area già scaricata in search.ipynb")

## 4. Mapping delle Coordinate ai Nodi più Vicini del Grafo

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Trova i nodi del grafo più vicini ai punti A e B
# ═══════════════════════════════════════════════════════════════
node_A = ox.nearest_nodes(G, X=store_lon, Y=store_lat)
node_B = ox.nearest_nodes(G, X=drop_lon,  Y=drop_lat)

# Coordinate dei nodi "snappati" sulla rete stradale
snap_A_lat = G.nodes[node_A]['y']
snap_A_lon = G.nodes[node_A]['x']
snap_B_lat = G.nodes[node_B]['y']
snap_B_lon = G.nodes[node_B]['x']

print(f"╔══════════════════════════════════════════════════════════════╗")
print(f"║  Punto A (originale) : ({store_lat:.6f}, {store_lon:.6f})")
print(f"║  Punto A (snappato)  : ({snap_A_lat:.6f}, {snap_A_lon:.6f})  [nodo {node_A}]")
print(f"╠══════════════════════════════════════════════════════════════╣")
print(f"║  Punto B (originale) : ({drop_lat:.6f}, {drop_lon:.6f})")
print(f"║  Punto B (snappato)  : ({snap_B_lat:.6f}, {snap_B_lon:.6f})  [nodo {node_B}]")
print(f"╚══════════════════════════════════════════════════════════════╝")

## 5. Esecuzione di A* sulla Rete Stradale Reale

In [5]:
# ═══════════════════════════════════════════════════════════════
# Euristica Haversine (distanza great-circle in metri)
# ═══════════════════════════════════════════════════════════════
def haversine(node_u, node_b, G_graph=G):
    """
    Calcola la distanza great-circle (Haversine) in metri
    tra due nodi del grafo, usando le loro coordinate (lat, lon).
    Euristica ammissibile: non sovrastima mai la distanza reale su strada.
    """
    lat1 = radians(G_graph.nodes[node_u]['y'])
    lon1 = radians(G_graph.nodes[node_u]['x'])
    lat2 = radians(G_graph.nodes[node_b]['y'])
    lon2 = radians(G_graph.nodes[node_b]['x'])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    R = 6_371_000  # raggio terrestre in metri
    return R * c

# ═══════════════════════════════════════════════════════════════
# Esecuzione A* con NetworkX
# ═══════════════════════════════════════════════════════════════
print("⏳ Esecuzione A* sulla rete stradale reale...")

try:
    route = nx.astar_path(G, source=node_A, target=node_B,
                          heuristic=haversine, weight='length')

    # Calcolo della distanza totale del percorso
    total_length_m = 0.0
    for i in range(len(route) - 1):
        u, v = route[i], route[i + 1]
        # Prendi l'arco più corto in caso di multi-grafo
        edge_data = G.get_edge_data(u, v)
        if edge_data:
            # In un MultiDiGraph, edge_data è un dict di dict
            min_length = min(d.get('length', 0) for d in edge_data.values())
            total_length_m += min_length

    total_length_km = total_length_m / 1000.0

    print(f"\n✅ Percorso trovato!")
    print(f"╔══════════════════════════════════════════════╗")
    print(f"║  Nodi nel percorso   : {len(route)}")
    print(f"║  Segmenti stradali   : {len(route) - 1}")
    print(f"║  Distanza totale     : {total_length_m:,.1f} m")
    print(f"║  Distanza totale     : {total_length_km:.2f} km")
    print(f"╚══════════════════════════════════════════════╝")

except nx.NetworkXNoPath:
    print("❌ Nessun percorso trovato tra i due punti!")
    route = []

NameError: name 'G' is not defined

## 6. Visualizzazione Statica con `ox.plot_graph_route`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Grafico statico: rete stradale + percorso A*
# ═══════════════════════════════════════════════════════════════
if route:
    fig, ax = ox.plot_graph_route(
        G, route,
        route_color='red',
        route_linewidth=4,
        route_alpha=0.8,
        node_size=0,
        bgcolor='white',
        edge_color='#999999',
        edge_linewidth=0.5,
        show=False,
        close=False,
        figsize=(12, 12)
    )

    # Marca Start (A) e Goal (B)
    ax.scatter(snap_A_lon, snap_A_lat, c='blue', s=150,
               zorder=5, edgecolors='black', linewidths=1.5, label='Store (A)')
    ax.scatter(snap_B_lon, snap_B_lat, c='green', s=150, marker='*',
               zorder=5, edgecolors='black', linewidths=1.5, label='Drop-off (B)')

    ax.set_title(
        f"A* su Rete Stradale Reale – {city_label}  |  "
        f"Distanza: {total_length_km:.2f} km  |  "
        f"Segmenti: {len(route) - 1}",
        fontsize=14, fontweight='bold', color='black'
    )
    ax.legend(loc='upper left', fontsize=11, framealpha=0.9)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Nessun percorso da visualizzare.")

## 7. Mappa Interattiva con Folium

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Mappa interattiva con Folium
# ═══════════════════════════════════════════════════════════════
if route:
    # Centro della mappa: punto medio tra A e B
    center_lat = (store_lat + drop_lat) / 2
    center_lon = (store_lon + drop_lon) / 2

    # Creazione mappa base
    m = folium.Map(location=[center_lat, center_lon],
                   zoom_start=14,
                   tiles='OpenStreetMap')

    # ── Percorso A* come polyline ──
    route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]
    folium.PolyLine(
        locations=route_coords,
        color='red',
        weight=5,
        opacity=0.8,
        popup=f"Percorso A* – {total_length_km:.2f} km"
    ).add_to(m)

    # ── Marker Store (Punto A) ──
    folium.Marker(
        location=[store_lat, store_lon],
        popup=folium.Popup(
            f"<b>🏪 Store (Punto A)</b><br>"
            f"Lat: {store_lat:.6f}<br>"
            f"Lon: {store_lon:.6f}<br>"
            f"Ordine: {sample['Order_ID']}",
            max_width=250
        ),
        icon=folium.Icon(color='blue', icon='home', prefix='fa')
    ).add_to(m)

    # ── Marker Drop-off (Punto B) ──
    folium.Marker(
        location=[drop_lat, drop_lon],
        popup=folium.Popup(
            f"<b>📦 Drop-off (Punto B)</b><br>"
            f"Lat: {drop_lat:.6f}<br>"
            f"Lon: {drop_lon:.6f}",
            max_width=250
        ),
        icon=folium.Icon(color='green', icon='flag', prefix='fa')
    ).add_to(m)

    # ── Salvataggio HTML ──
    html_path = f"astar_route_{city_label.lower()}.html"
    m.save(html_path)
    print(f"✅ Mappa interattiva {city_label} salvata in: {html_path}")

    # Visualizzazione inline nel notebook
    display(m)
else:
    print("⚠️ Nessun percorso da visualizzare sulla mappa.")

## 8. Statistiche del Percorso

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Riepilogo statistiche del percorso
# ═══════════════════════════════════════════════════════════════
if route:
    # Distanza in linea d'aria (Haversine) tra A e B
    straight_line_m = haversine(node_A, node_B)
    straight_line_km = straight_line_m / 1000.0

    # Fattore di deviazione
    detour_factor = total_length_km / straight_line_km if straight_line_km > 0 else float('inf')

    # Tempo stimato di percorrenza
    AVG_SPEED_KMH_CITY = 25  # velocità media urbana in India (km/h)
    travel_time_min = (total_length_km / AVG_SPEED_KMH_CITY) * 60

    print(f"╔══════════════════════════════════════════════════════════════════╗")
    print(f"║                   RIEPILOGO PERCORSO A* – {city_label.upper()}                    ║")
    print(f"╠══════════════════════════════════════════════════════════════════╣")
    print(f"║  Coordinate partenza (Store)     : ({store_lat:.6f}, {store_lon:.6f})")
    print(f"║  Coordinate arrivo  (Drop-off)   : ({drop_lat:.6f}, {drop_lon:.6f})")
    print(f"╠══════════════════════════════════════════════════════════════════╣")
    print(f"║  Distanza su strada (A*)         : {total_length_km:.2f} km")
    print(f"║  Distanza in linea d'aria         : {straight_line_km:.2f} km")
    print(f"║  Fattore di deviazione            : {detour_factor:.2f}x")
    print(f"╠══════════════════════════════════════════════════════════════════╣")
    print(f"║  Nodi attraversati               : {len(route)}")
    print(f"║  Segmenti stradali               : {len(route) - 1}")
    print(f"║  Tempo stimato ({AVG_SPEED_KMH_CITY} km/h)          : {travel_time_min:.1f} min")
    print(f"╚══════════════════════════════════════════════════════════════════╝")

    # ── Tabella riassuntiva con pandas ──
    summary = pd.DataFrame({
        'Metrica': [
            'Distanza su strada (km)',
            'Distanza linea d\'aria (km)',
            'Fattore di deviazione',
            'Nodi nel percorso',
            'Segmenti stradali',
            f'Tempo stimato @ {AVG_SPEED_KMH_CITY} km/h (min)',
            'Latitudine partenza',
            'Longitudine partenza',
            'Latitudine arrivo',
            'Longitudine arrivo'
        ],
        'Valore': [
            f"{total_length_km:.2f}",
            f"{straight_line_km:.2f}",
            f"{detour_factor:.2f}",
            len(route),
            len(route) - 1,
            f"{travel_time_min:.1f}",
            f"{store_lat:.6f}",
            f"{store_lon:.6f}",
            f"{drop_lat:.6f}",
            f"{drop_lon:.6f}"
        ]
    })
    display(summary.style.set_properties(**{'text-align': 'left'})
                         .set_caption(f"📊 Statistiche Percorso A* – {city_label} (Rete Stradale Reale)")
                         .hide(axis='index'))
else:
    print("⚠️ Nessun percorso calcolato.")